# Test du scraping

In [1]:
from bs4 import BeautifulSoup

In [2]:
from src.backend.scraping import fetch_page, enrich_with_scraping
from src.backend.processing import clean_parcoursup_data

url = "https://dossierappel.parcoursup.fr/Candidats/public/fiches/afficherFicheFormation?g_ta_cod=10&typeBac=0&originePc=0"

html = fetch_page(url)
len(html), html[:500]

(109029,
 '\r\n\r\n\r\n\r\n\r\n\r\n\r\n\r\n\t\r\n\t\r\n\t\r\n\t\r\n\t\t\r\n\r\n\r\n\r\n\r\n\r\n\r\n\r\n\r\n\r\n\r\n\r\n\r\n\r\n<!DOCTYPE html>\r\n<html lang="fr">\r\n\t\r\n\t<head>\r\n\t\t\r\n\t\t<title>Fiche formation - Concours Geipi Polytech Formation d\'ingénieur Bac + 5 - Bac général - Polytech Dijon (21) - Parcoursup</title>\r\n\t\t<meta charset="UTF-8">\r\n\t\t<meta name="format-detection" content="telephone=no,date=no,address=no,email=no,url=no">\r\n\t\t\r\n\t\t\r\n\t\t\r\n\t\t\t<meta name="viewport" content="width=device-width, initial-scale=1">\r\n\t\t\t\r\n\t\t\r\n\t\t\r\n\t\t\r\n\t\t\r\n\t\t<script crossorigin="anony')

## Test de récupération de la page

In [3]:
from importlib import reload
from src.backend import scraping

reload(scraping)  # recharge le module après modification

from src.backend.scraping import scrape_formation

infos = scrape_formation(url)
infos

{'presentation': "Le réseau Polytech regroupe 16 écoles publiques délivrant le diplôme d'ingénieur reconnu par la Commission des Titres d'Ingénieur. La formation se déroule en 5 ans, avec un cycle préparatoire généraliste de 2 ans s'articulant autour de matières scientifiques fondamentales associées à une démarche ingénieur.\nCe cycle préparatoire sciences et techniques de l’ingénieur permet ensuite de s'orienter dans l'une des 100 spécialités d’ingénieur du réseau en restant dans la même école ou en changeant d’école entre le cycle préparatoire et le cycle ingénieur. Les 100 spécialités sont réparties en 12 domaines (\nhttp://www.polytech-reseau.org/se-former-avec-polytech/\n) : Eau, environnement, aménagement / Électronique et systèmes numériques / Énergétique, génie des procédés / Génie biologique et alimentaire / Génie biomédical, instrumentation / Génie civil / Génie industriel / Informatique / Matériaux.\nLes spécialités d’ingénieur sont accessibles en statut d’étudiant et pour l

## Affichage des titres

In [4]:
soup = BeautifulSoup(html, "html.parser")

# Afficher tous les titres h2/h3 pour repérer les sections
[t.get_text(strip=True) for t in soup.find_all(["h2", "h3", "h4"])]

["Concours Geipi PolytechFormation d'ingénieur Bac + 5 - Bac général",
 'Mon appréciation personnelle',
 'Polytech Dijon (21)',
 'Présentation de la formation',
 'À savoir',
 'L’examen des candidatures',
 "Grille d’analyse des candidatures définie par la commission d'examen des voeux de la formation",
 'Détails de la grille d’analyse des candidatures par la commission',
 'Les attendus nationaux',
 'Les conditions pour candidater',
 'Les épreuves de sélection',
 'Frais de candidature',
 "Les chiffres globaux d'accès à cette formation en 2025",
 "En savoir plus sur l'accès à la formationselon mon profil",
 'Action mise en œuvre pour favoriser l’égalité d’accès dans l’enseignement supérieur',
 "Poursuite d'études",
 'Débouchés professionnels',
 'Établissement',
 'Rechercher une personne avec qui échanger']

## Récupération d'informations pertinentes

In [5]:
from src.backend.scraping import fetch_page, parse_formation_page

infos = parse_formation_page(html)
infos["presentation"][:800]  # pour voir le début

"Le réseau Polytech regroupe 16 écoles publiques délivrant le diplôme d'ingénieur reconnu par la Commission des Titres d'Ingénieur. La formation se déroule en 5 ans, avec un cycle préparatoire généraliste de 2 ans s'articulant autour de matières scientifiques fondamentales associées à une démarche ingénieur.\nCe cycle préparatoire sciences et techniques de l’ingénieur permet ensuite de s'orienter dans l'une des 100 spécialités d’ingénieur du réseau en restant dans la même école ou en changeant d’école entre le cycle préparatoire et le cycle ingénieur. Les 100 spécialités sont réparties en 12 domaines (\nhttp://www.polytech-reseau.org/se-former-avec-polytech/\n) : Eau, environnement, aménagement / Électronique et systèmes numériques / Énergétique, génie des procédés / Génie biologique et alimen"

In [6]:
html = fetch_page(url)
data = parse_formation_page(html)
data["criteres_entree"]

"Disposer de compétences en matière de communication numérique et d’expression écrite et orale afin de pouvoir défendre un argumentaire précis et présenter un projet.\nDisposer de compétences écrites et orales en langues étrangères, au minimum en anglais afin d’être capable de mener des recherches documentaires, de travailler à partir de documents originaux.\nDisposer d’une bonne culture générale, faire preuve d’ouverture d’esprit et de motivation pour les enjeux sociétaux.\n\nDisposer de solides compétences scientifiques, technologiques et les démarches\xa0associées\nMener individuellement des réflexions et des recherches pour un travail approfondi à partir de connaissances et compétences acquises\nSavoir travailler de manière autonome ou en groupe pour mener à bien ses apprentissages\nMontrer des dispositions à la conduite de projets scientifiques et technologiques\nDisposer des compétences de réflexion, d'argumentation et d'expression en français et en\xa0langues vivantes."

In [7]:
# Voir tous les h5
for h5 in soup.find_all("h5"):
    print("H5:", repr(h5.get_text(strip=True)))

H5: 'Frais de scolarité'
H5: "Information : Bourses de l'enseignement supérieur"
H5: 'Apprentissage'
H5: 'Langues et options'
H5: 'Aménagement pour les publics ayant un profil particulier'
H5: "Dispositifs d'aide à la réussite"
H5: 'Qui analyse les candidatures ?'
H5: 'Comment se déroule l’analyse des candidatures ?'
H5: 'Rapport public'
H5: 'Conseils aux candidats'
H5: 'Les attendus complémentaires'
H5: 'Voie générale'
H5: 'Épreuves écrites'
H5: 'Dates des épreuves'
H5: "Voir la présentation des épreuves sur le site de l'établissement"
H5: 'Frais Concours Geipi Polytech'
H5: 'Informations supplémentaires'
H5: "Sur l'accompagnement pour les situations de handicap"
H5: 'Sur la formation et son organisation'
H5: 'Echanger avec un étudiant ambassadeur de la formation'


In [8]:
for h5 in soup.find_all(lambda tag: tag.name == "h5" and "Les attendus complémentaires" in tag.get_text()):
    print("TROUVÉ H5:", repr(h5.get_text(strip=True)))
    print("Sibling direct:", h5.find_next_sibling())

TROUVÉ H5: 'Les attendus complémentaires'
Sibling direct: <div class="word-break-break-word"><ol><li>Disposer de solides compétences scientifiques, technologiques et les démarches associées</li><li>Mener individuellement des réflexions et des recherches pour un travail approfondi à partir de connaissances et compétences acquises </li><li>Savoir travailler de manière autonome ou en groupe pour mener à bien ses apprentissages</li><li>Montrer des dispositions à la conduite de projets scientifiques et technologiques</li><li>Disposer des compétences de réflexion, d'argumentation et d'expression en français et en langues vivantes.</li></ol></div>


In [9]:
# Voir tous les h5
for h4 in soup.find_all("h4"):
    print("H4:", repr(h4.get_text(strip=True)))

H4: 'Présentation de la formation'
H4: 'À savoir'
H4: 'L’examen des candidatures'
H4: "Grille d’analyse des candidatures définie par la commission d'examen des voeux de la formation"
H4: 'Détails de la grille d’analyse des candidatures par la commission'
H4: 'Les attendus nationaux'
H4: 'Les conditions pour candidater'
H4: 'Les épreuves de sélection'
H4: 'Frais de candidature'
H4: "Les chiffres globaux d'accès à cette formation en 2025"
H4: "En savoir plus sur l'accès à la formationselon mon profil"
H4: 'Action mise en œuvre pour favoriser l’égalité d’accès dans l’enseignement supérieur'
H4: "Poursuite d'études"
H4: 'Débouchés professionnels'
H4: 'Établissement'
H4: 'Rechercher une personne avec qui échanger'


In [10]:
html = fetch_page(url)
data = parse_formation_page(html)
data["debouches_professionnels"]

"La formation scientifique, technique, économique et humaine des ingénieurs Polytech, les expériences en entreprise, la culture de l’international et de la recherche, et les contacts directs avec les professionnels via les cours et les projets, permettent aux ingénieurs une insertion professionnelle rapide\xa0:\nPlus de 90%\ndes diplômés Polytech ont un emploi en moins de 3 mois après la sortie d’école (entreprises de toutes tailles en France et à l'étranger).\nLes diplômés Polytech exercent dans des secteurs variés\xa0: sociétés de conseil, technologie de l'information, automobile, aéronautique, navale et ferroviaire, BTP, construction, énergie, environnement, agroalimentaire, assurance, institutions financières....\nLes fonctions assurées par les ingénieurs en activité couvrent de nombreux domaines\xa0: recherche et développement, études, conception, prototypage, production, support technique, maintenance, recyclage, gestion de produit, gestion d'affaires, consultance, qualité, resso

In [11]:
html = fetch_page(url)
data = parse_formation_page(html)
print(data["poursuite_etudes"])
print(data["frais_scolarite"])
print(data["frais_scolarite_boursiers"])
print(data["langues_options"])

La formation scientifique, technique, économique et humaine des ingénieurs Polytech couplée aux expériences en entreprise, à l’international, dans les laboratoires recherche offrent une grande variété de poursuite d’étude. A l'issue du cycle ingénieur, vous pourrez commencer votre carrière professionnelle en rejoignant une entreprise, en créant une entreprise ou en poursuivant vos études pour spécialiser ou diversifier vos compétences avec une thèse de doctorat ou un master en France ou à étranger.  Les ingénieurs peuvent également se spécialiser dans un domaine particulier via des années de spécialisation (bac+6) au sien du Réseau Polytech notamment.
628 €
0
Langue vivante 1 :
Anglais
Langue vivante 2 :
Allemand, Chinois, Espagnol, Italien, Japonais, Polonais, Portugais, Roumain et  Russe
Niveau de français requis pour s'inscrire à la formation : B2


In [12]:
print(data["nb_places"])
print(data["diplome_controle_par_etat"])
print(data["formation_selective"])
print(data["formation_ouverte_boursiers"])

120
1
1
1


In [13]:
print(data["epreuves_selection"])

Tous les candidats inscrits au concours seront convoqués à une épreuve écrite
de 3 heures, le
mardi 28 avril après-midi.
L'épreuve écrite comportera :
Un sujet de mathématiques (1h) : QCM basé sur le programme commun à la spécialité mathématiques et à l’option maths complémentaires
Deux sujets à choisir parmi : mathématiques, physique-chimie, numérique et sciences informatiques, SVT/biologie-écologie ou sciences de l'ingénieur
Les sujets de l'épreuve écrite 2026 porteront majoritairement sur le programme de Terminale. Les programmes des révisions pour l'épreuve écrite sont disponibles sur le site du Concours Geipi Polytech dans l'onglet “téléchargement”.


In [14]:
print(data["frais_candidature"])
print(data["frais_candidature_boursiers"])

60,00 €
0,00 €


# Test du scraping et ajout des colonnes sur échantillon

In [17]:
from pathlib import Path

csv_path = Path("../data/raw/fr-esr-cartographie_formations_parcoursup.csv")
df_clean = clean_parcoursup_data(csv_path, "2026")
df_clean.head()

,session,id_etablissement,name_etablissement,type_etablissement,type_formation,name_formation,mentions_specialites,apprentissage,internat,amenagement,...,website_etablissement,short_name_formation,is_apprentissage,internat_code,is_presentiel,is_partiel_distance,is_full_distance,has_sport_amenagement,has_artist_amenagement,has_other_amenagement
0,2026,0212108C,Polytech Dijon (21),Publics,Formations des écoles d'ingénieurs,Formation d'ingénieur Bac + 5 - Bac général,Formation d'ingénieur Bac + 5,NaN,NaN,"Enseignement en présentiel,Formations avec amé...",...,http://esirem.u-bourgogne.fr,FI-BAC5,0,0,1,0,0,1,1,0
1,2026,0492226D,Polytech Angers (49),Publics,Formations des écoles d'ingénieurs,Formation d'ingénieur Bac + 5 - Bac général,Formation d'ingénieur Bac + 5,NaN,NaN,"Enseignement en présentiel,Formations avec amé...",...,http://www.polytech-angers.fr,FI-BAC5,0,0,1,0,0,1,0,0
2,2026,0542259M,EEIGM Nancy - Groupe INP (54),Publics,Formations des écoles d'ingénieurs,Formation d'ingénieur Bac + 5 - Bac général,Formation d'ingénieur Bac + 5,NaN,NaN,"Enseignement en présentiel,Formations avec amé...",...,http://eeigm.univ-lorraine.fr/,FI-BAC5,0,0,1,0,0,1,1,0
3,2026,0542307P,ENSGSI Nancy - Groupe INP (54),Publics,Formations des écoles d'ingénieurs,Formation d'ingénieur Bac + 5 - Bac général,Formation d'ingénieur Bac + 5,NaN,NaN,"Enseignement en présentiel,Formations avec amé...",...,https://ensgsi.univ-lorraine.fr/,FI-BAC5,0,0,1,0,0,1,1,1
4,2026,0597060D,IMT Nord Europe (Villeneuve-d'Ascq - 59),Publics,Formations des écoles d'ingénieurs,Formation d'ingénieur Bac + 5 - Bac général,Formation d'ingénieur Bac + 5,NaN,NaN,"Enseignement en présentiel,Formations avec amé...",...,https://www.imt-nord-europe.fr,FI-BAC5,0,0,1,0,0,1,1,0


In [18]:
df_enriched = enrich_with_scraping(df_clean, url_col="link_formation")
df_enriched.head()

LIGNE 0 URL = https://dossierappel.parcoursup.fr/Candidats/public/fiches/afficherFicheFormation?g_ta_cod=10&typeBac=0&originePc=0
  -> clés infos: ['presentation', 'criteres_entree', 'debouches_professionnels', 'poursuite_etudes', 'frais_scolarite', 'frais_scolarite_boursiers', 'langues_options', 'nb_places', 'formation_ouverte_boursiers', 'diplome_controle_par_etat', 'formation_selective', 'epreuves_selection', 'frais_candidature', 'frais_candidature_boursiers']
    écrit colonne presentation
    écrit colonne criteres_entree
    écrit colonne debouches_professionnels
    écrit colonne poursuite_etudes
    écrit colonne frais_scolarite
    écrit colonne frais_scolarite_boursiers
    écrit colonne langues_options
    écrit colonne nb_places
    écrit colonne formation_ouverte_boursiers
    écrit colonne diplome_controle_par_etat
    écrit colonne formation_selective
    écrit colonne epreuves_selection
    écrit colonne frais_candidature
    écrit colonne frais_candidature_boursiers
LI

,session,id_etablissement,name_etablissement,type_etablissement,type_formation,name_formation,mentions_specialites,apprentissage,internat,amenagement,...,frais_scolarite,frais_scolarite_boursiers,langues_options,nb_places,formation_ouverte_boursiers,diplome_controle_par_etat,formation_selective,epreuves_selection,frais_candidature,frais_candidature_boursiers
0,2026,0212108C,Polytech Dijon (21),Publics,Formations des écoles d'ingénieurs,Formation d'ingénieur Bac + 5 - Bac général,Formation d'ingénieur Bac + 5,NaN,NaN,"Enseignement en présentiel,Formations avec amé...",...,628 €,0,Langue vivante 1 :\nAnglais\nLangue vivante 2 ...,120.0,1.0,1.0,1.0,Tous les candidats inscrits au concours seront...,"60,00 €","0,00 €"
1,2026,0492226D,Polytech Angers (49),Publics,Formations des écoles d'ingénieurs,Formation d'ingénieur Bac + 5 - Bac général,Formation d'ingénieur Bac + 5,NaN,NaN,"Enseignement en présentiel,Formations avec amé...",...,628 €,0,Langue vivante 1 :\nAnglais\nLangue vivante 2 ...,95.0,1.0,1.0,1.0,Tous les candidats inscrits au concours seront...,"60,00 €","0,00 €"
2,2026,0542259M,EEIGM Nancy - Groupe INP (54),Publics,Formations des écoles d'ingénieurs,Formation d'ingénieur Bac + 5 - Bac général,Formation d'ingénieur Bac + 5,NaN,NaN,"Enseignement en présentiel,Formations avec amé...",...,628 €,0,Langue vivante 1 :\nAnglais\nLangue vivante 2 ...,82.0,1.0,1.0,1.0,Tous les candidats inscrits au concours seront...,"60,00 €","0,00 €"
3,2026,0542307P,ENSGSI Nancy - Groupe INP (54),Publics,Formations des écoles d'ingénieurs,Formation d'ingénieur Bac + 5 - Bac général,Formation d'ingénieur Bac + 5,NaN,NaN,"Enseignement en présentiel,Formations avec amé...",...,628 €,0,Langue vivante 1 :\nAnglais\nLangue vivante 2 ...,50.0,1.0,1.0,1.0,Tous les candidats inscrits au concours seront...,"60,00 €","0,00 €"
4,2026,0597060D,IMT Nord Europe (Villeneuve-d'Ascq - 59),Publics,Formations des écoles d'ingénieurs,Formation d'ingénieur Bac + 5 - Bac général,Formation d'ingénieur Bac + 5,NaN,NaN,"Enseignement en présentiel,Formations avec amé...",...,Français et UE : 2650 € / an - Hors UE : 4850 ...,0,Langue vivante 1 :\nAnglais\nLangue vivante 2 ...,160.0,1.0,1.0,1.0,Tous les candidats inscrits au concours seront...,"60,00 €","0,00 €"
